In [0]:
dbutils.library.restartPython()

In [0]:

# COMMAND ----------
# Define Widgets for User Input
dbutils.widgets.text("snapshot", "202505", "Snapshot (YYYYMM)")
dbutils.widgets.text("DIL", "", "DIL Value (Optional)")
dbutils.widgets.text("product", "", "Product (Optional)")

# Retrieve Parameters
SNAPSHOT = dbutils.widgets.get("snapshot")
DIL = dbutils.widgets.get("DIL")
PRODUCT = dbutils.widgets.get("product")

# COMMAND ----------
import os
import sys
import logging
import yaml
from argparse import Namespace

# Add the repo's src directory to the Python path to allow imports
# This assumes the notebook is located in amfs_tm/src/
REPO_ROOT = os.path.abspath("/Workspace/Users/hanif.m.abidin@gmail.com/amfs_telemarketing_new_acquisition")
if REPO_ROOT not in sys.path:
    sys.path.append(REPO_ROOT)

from lib.clean.clean_all import clean_all
from lib.feature.features_generation import features_generation
from lib.filters.filter_apply import filter_apply
from lib.matrix.merge_all_features import create_modeling_matrix
from lib.inference.predictor import run_prediction

# Setup Logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# COMMAND ----------
# Define Configuration Paths
# Based on the migrated repository structure
CONFIG_DIR = os.path.join(REPO_ROOT, "configs")
MAIN_CONFIG = os.path.join(CONFIG_DIR, "main_config.yaml")
CLEAN_CONFIG = os.path.join(CONFIG_DIR, "clean_config.yaml")
FEATURES_CONFIG = os.path.join(CONFIG_DIR, "features_config.yaml")
FILTERS_CONFIG = os.path.join(CONFIG_DIR, "filters_config.yaml")
MATRIX_CONFIG = os.path.join(CONFIG_DIR, "matrix_config.yaml")
PREDICT_CONFIG = os.path.join(CONFIG_DIR, "predict_config.yaml")

# Create a mock args object to pass to the library functions
args = Namespace(snapshot=SNAPSHOT, dil=DIL, product=PRODUCT)


# COMMAND ----------
# DBTITLE 1,Step 1: Data Cleaning
# Processes raw files into a standardized format
logging.info(f"Starting Data Cleaning for snapshot {SNAPSHOT}...")
# Note: Ensure paths in clean_config.yaml are updated to Volume paths
clean_all(MAIN_CONFIG, CLEAN_CONFIG, args)

# DBTITLE 1,Step 2: Feature Generation
logging.info(f"Starting Feature Generation...")
features_generation(MAIN_CONFIG, FEATURES_CONFIG, args)

# DBTITLE 1,Step 3: Business Rules Filtering
logging.info(f"Created filtered cifno based on business rules...")
filter_apply(MAIN_CONFIG, FILTERS_CONFIG, args)

# DBTITLE 1,Step 4: Matrix Creation
# Merges all features and the base population into a modeling matrix
logging.info(f"Merging features into modeling matrix...")
create_modeling_matrix(MAIN_CONFIG, MATRIX_CONFIG, args)

# COMMAND ----------
# DBTITLE 1,Step 5: Propensity Scoring (Inference)
# This will load the pickle, score the matrix, and save results to 05_campaigns
logging.info(f"Run prediction...")
run_prediction(MAIN_CONFIG, PREDICT_CONFIG, args)